# Phase 8 - Encoder generalization (GATv2)

Does the framework's decision survive a change of aggregator? GraphSAGE stays the study's encoder; nothing here re-fits a frozen rule.

Only the convolution varies: `SAGEConv(mean)` -> `GATv2Conv(4 heads)`. Graph, role graph, features, split, objective, epochs and eval are identical.

Link prediction, K=10, seeds 42-44, seven variants, paired band, 30 graphs (`cfg.ENCODER_PANEL`).

In [ ]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))
import pandas as pd
from experiments import encoder_transfer as et

STYLE = [{"selector": "table", "props": [("font-size", "12px")]},
         {"selector": "th", "props": [("text-align", "center")]},
         {"selector": "td", "props": [("text-align", "center")]}]
show = lambda df: display(df.style.set_table_styles(STYLE).hide(axis="index"))

fourcell = pd.read_csv("results/encoder_transfer_fourcell.csv")
agree = pd.read_csv("results/encoder_transfer_agreement.csv")
perseed = pd.read_csv("results/encoder_transfer_perseed.csv")
PANEL = sorted(agree.dataset)
cross = et.cross(perseed, "gatv2_edge", "graphsage_edge").set_index("dataset")
gv = fourcell[fourcell.encoder == "gatv2_edge"].set_index("dataset")
w = fourcell.pivot(index="dataset", columns="encoder", values=["original", "best_aug", "gap", "aug_wins"])
w.columns = [f"{b.replace('_edge','')}_{a}" for a, b in w.columns]
print(f"{len(PANEL)} graphs")

## Q1 · Does augmentation still help when the encoder changes?

The four cells: `GraphSAGE`, `GraphSAGE+aug`, `GATv2`, `GATv2+aug`. `+aug` = the best non-`original` variant per graph and encoder.

In [ ]:
tbl = pd.DataFrame({"original": [w.graphsage_original.mean(), w.gatv2_original.mean()],
                    "+aug":     [w.graphsage_best_aug.mean(), w.gatv2_best_aug.mean()]},
                   index=["GraphSAGE", "GATv2"]).round(4)
tbl["aug - original"] = (tbl["+aug"] - tbl["original"]).round(4)
tbl["aug wins"] = [f"{int(w.graphsage_aug_wins.sum())}/{len(w)}", f"{int(w.gatv2_aug_wins.sum())}/{len(w)}"]
print(f"MEAN AUC, {len(w)} graphs")
display(tbl)

best = w[["graphsage_original", "graphsage_best_aug", "gatv2_original", "gatv2_best_aug"]].idxmax(axis=1).value_counts()
print(f"\nBEST OF THE FOUR, counted over {len(w)} graphs")
display(pd.DataFrame([[best.get("graphsage_original", 0), best.get("graphsage_best_aug", 0)],
                      [best.get("gatv2_original", 0), best.get("gatv2_best_aug", 0)]],
                     index=["GraphSAGE", "GATv2"], columns=["original", "+aug"]))

**A1.** Augmentation helps both encoders on average, but on far fewer graphs under GATv2 - **13/30 vs 21/30**. GATv2 takes the best of the four cells on most graphs.

## Q2 · Can GATv2 alone already beat GraphSAGE+aug?

GATv2 on the **original** graph vs GraphSAGE on its **best augmented** graph, paired on the shared split. A strict beat - a tie is not a win.

In [ ]:
band = cross.paired_ratio.apply(lambda r: "GATv2 alone beats" if r > 1 else
                                ("no difference" if abs(r) <= 1 else "GATv2 alone loses"))
rows = []
for k in ["GATv2 alone beats", "no difference", "GATv2 alone loses"]:
    n = int((band == k).sum())
    helps = int(gv.loc[sorted(band[band == k].index)].aug_wins.sum())
    rows.append({"GATv2 alone vs GraphSAGE+aug": k, "graphs": f"{n} / {len(PANEL)}",
                 "of those, augmentation helps GATv2": f"{helps} / {n}"})
show(pd.DataFrame(rows))

**A2.** Yes on 14/30 - so the objection holds about half the time. But the two outcomes are near-disjoint: where GATv2 alone already wins, augmentation adds almost nothing (2 of 14); where GATv2 alone loses, augmentation is what rescues it (8 of 9).

**Augmentation and a stronger aggregator are substitutes, not complements.**

## Q3 · On the graphs where GATv2 alone fails, can augmentation rescue it?

The 9 graphs from Q2 where GATv2 on the original graph loses to GraphSAGE+aug.

In [ ]:
fail = sorted(band[band == "GATv2 alone loses"].index)
g = gv.loc[fail]
show(pd.DataFrame([
    {"where GATv2 alone fails": "graphs",                         "value": f"{len(g)} / {len(PANEL)}"},
    {"where GATv2 alone fails": "GATv2 alone, mean AUC",           "value": f"{g.original.mean():.4f}"},
    {"where GATv2 alone fails": "GATv2 + augmentation, mean AUC",  "value": f"{g.best_aug.mean():.4f}"},
    {"where GATv2 alone fails": "mean gain",                       "value": f"{(g.best_aug - g.original).mean():+.4f}"},
    {"where GATv2 alone fails": "graphs rescued",                  "value": f"{int(g.aug_wins.sum())} / {len(g)}"}]))

**A3.** Yes - augmentation rescues 8 of the 9, lifting mean AUC from 0.5072 to 0.6619.

## Q4 · Do the framework's calls survive?

Stage 2 is conditional on stage 1 saying *augment*, so it is only scorable where both encoders augment and both resolve a single signal.

In [ ]:
dis = agree[~agree.stage1_agree]
c = agree[agree.stage2_comparable]
both = c[(c.stage1_baseline == "augment") & (c.stage1_encoder == "augment")]
show(pd.DataFrame([
    {"call": "stage 1 - keep or augment", "GraphSAGE and GATv2 agree": f"{int(agree.stage1_agree.sum())} / {len(agree)}",
     "note": f"all {len(dis)} disagreements run augment -> keep; none the other way"},
    {"call": "stage 2 - which signal", "GraphSAGE and GATv2 agree": f"{int(both.stage2_agree.sum())} / {len(both)}",
     "note": "scored only where both encoders augment"}]))

**A4.** Stage 1 survives 22/30, and every failure is one-directional - GATv2 makes augmentation unnecessary, never newly necessary. Stage 2 survives 5/7.

## Caveats

- **GATv2 runs on GraphSAGE's hyperparameters** (lr 0.01, 50 epochs, 2 layers, 64 dims). Deliberate - tuning would break the only-the-convolution-changes contract - so a GATv2 loss never means attention is worse, only worse at these settings.
- **Are the added edges really long-range?** For every edge the role graph adds, we measured how many hops apart its two nodes are in the original graph (`experiments/edge_distance.py`, 6 graphs, K=10): median 3-6 hops, and only 14% are 1-2 hops apart. So they are not short links that attention on the original graph could already reach.